# AI Digest — A Real Google ADK News Agent

**Kaggle AI Agents Capstone**

A single agent that discovers AI/ML news, then *reasons* with Google's Gemini
model to rank and explain the top 10 stories — and *critiques its own output*
in a self-review loop before publishing.

This notebook uses the **official Agent Development Kit (ADK 2.0)**:

- `google.adk.Agent` + `Runner` + `InMemorySessionService` (real agent runtime)
- `FunctionTool` the model chooses to call (`fetch_ai_news`)
- Gemini via the modern `google.genai` SDK (model: `gemini-flash-latest`)
- A **self-critique loop**: a critic agent reviews the brief and the curator
  revises it — genuine reasoning, not a fixed pipeline
- Deterministic keyword ranking only as an offline **fallback**

> Runs end-to-end with a `GEMINI_API_KEY`. Without one (or without network),
> it degrades gracefully to the keyword fallback and still emits a valid brief.


In [ ]:
# ── Setup: install ADK + genai, load key, init Gemini client ─────────────────
# On Kaggle this installs the real packages; locally it is a no-op if present.
try:
    import google.adk  # noqa: F401
    import google.genai  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "google-adk", "google-genai", "nest-asyncio"],
        check=False,
    )

import os
import json

# gemini-flash-latest is the current, verified-working alias for this key.
GEMINI_MODEL = "gemini-flash-latest"


def _load_api_key() -> str:
    """Load GEMINI_API_KEY from Kaggle Secrets, then the environment."""
    try:
        from kaggle_secrets import UserSecretsClient
        key = (UserSecretsClient().get_secret("GEMINI_API_KEY") or "").strip()
        if key:
            print("✅ Loaded GEMINI_API_KEY from Kaggle Secrets")
            return key
        print("⚠️  Kaggle secret 'GEMINI_API_KEY' is attached but empty")
    except ImportError:
        pass  # Not on Kaggle — fall through to the environment.
    except Exception as e:
        # Surface the real reason (e.g. secret not attached to THIS session).
        print(f"⚠️  Kaggle Secrets lookup failed: {type(e).__name__}: {str(e)[:120]}")
        print("   → In the Kaggle editor: attach the secret, then use "
              "'Run All' / restart the session so it is loaded.")
    key = os.getenv("GEMINI_API_KEY", "").strip()
    print("✅ Loaded GEMINI_API_KEY from environment" if key
          else "⚠️  No GEMINI_API_KEY found — will use keyword fallback")
    return key


GEMINI_API_KEY = _load_api_key()
# ADK reads GOOGLE_API_KEY; genai client reads the key we pass explicitly.
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ.setdefault("GOOGLE_GENAI_USE_ENTERPRISE", "FALSE")

# Single genai client, shared by every LLM call in this notebook.
genai_client = None
if GEMINI_API_KEY:
    try:
        from google import genai
        genai_client = genai.Client(api_key=GEMINI_API_KEY)
    except Exception as e:
        print(f"⚠️  Could not init genai client: {str(e)[:80]}")

LLM_ENABLED = genai_client is not None
print(f"LLM path: {'Gemini (' + GEMINI_MODEL + ')' if LLM_ENABLED else 'keyword fallback'}")


## Data models

Type-safe dataclasses enforce the output contract at runtime: valid URLs, a
rank in 1–10, and exactly 10 cards per brief. If the agent ever produces a
malformed brief, construction fails loudly instead of shipping bad data.


In [ ]:
# ── Models (dataclasses, stdlib only) ────────────────────────────────────────
from datetime import date
from typing import List
from dataclasses import dataclass


@dataclass
class NewsItem:
    source_id: str
    title: str
    url: str
    summary: str = ""

    def __post_init__(self):
        if not self.url.startswith(("http://", "https://")):
            raise ValueError(f"Invalid URL: {self.url}")


@dataclass
class BriefCard:
    rank: int
    title: str
    url: str
    why_it_matters: str

    def __post_init__(self):
        if not (1 <= self.rank <= 10):
            raise ValueError(f"Rank must be 1-10, got {self.rank}")
        if not self.url.startswith("https://"):
            raise ValueError(f"URL must be HTTPS: {self.url}")


@dataclass
class DailyBrief:
    date: str
    theme: str
    cards: List[BriefCard]
    schema_version: str = "1.0"

    def __post_init__(self):
        if len(self.cards) != 10:
            raise ValueError(f"Must have exactly 10 cards, got {len(self.cards)}")


print("✅ Models initialized")


## Tool 1 — Discover (real arXiv RSS)

The discovery step fetches live papers from arXiv's `cs.AI` and `cs.LG` feeds
using only the standard library. If the network is unavailable, it falls back
to a fixed sample set so the rest of the notebook still runs. This function is
wrapped as an **ADK `FunctionTool`** further down, so the agent itself decides
to call it.


In [ ]:
# ── RSS Parser + discovery (stdlib only) ─────────────────────────────────────
import xml.etree.ElementTree as ET
import urllib.request


def parse_rss_bytes(data: bytes, source_id: str, limit: int = 20) -> List[NewsItem]:
    """Parse RSS 2.0 / Atom feeds using stdlib only."""
    try:
        root = ET.fromstring(data)
    except ET.ParseError:
        return []

    items = []
    _ATOM = "http://www.w3.org/2005/Atom"
    _CONTENT = "{http://purl.org/rss/1.0/modules/content/}encoded"

    def _text(el):
        return (el.text or "").strip() if el is not None else ""

    channel = root.find("channel")
    if channel is not None:  # RSS 2.0
        for el in channel.findall("item"):
            title = _text(el.find("title"))
            url = _text(el.find("link"))
            summary = _text(el.find("description")) or _text(el.find(_CONTENT))
            if title and url and url.startswith("http"):
                try:
                    items.append(NewsItem(source_id, title, url, summary[:500]))
                except ValueError:
                    pass
            if len(items) >= limit:
                break
        return items

    for el in root.findall(f"{{{_ATOM}}}entry"):  # Atom
        title = _text(el.find(f"{{{_ATOM}}}title"))
        url = ""
        for link in el.findall(f"{{{_ATOM}}}link"):
            href = link.get("href", "")
            if link.get("rel", "alternate") in ("alternate", "") and href.startswith("http"):
                url = href
                break
        summary = _text(el.find(f"{{{_ATOM}}}summary")) or _text(el.find(f"{{{_ATOM}}}content"))
        if title and url:
            try:
                items.append(NewsItem(source_id, title, url, summary[:500]))
            except ValueError:
                pass
        if len(items) >= limit:
            break
    return items


def _fallback_items() -> List[NewsItem]:
    """Sample AI/ML items used only when arXiv is unreachable."""
    raw = [
        ("Llama 3.1 405B Reaches State-of-the-Art Performance", "2407.21022",
         "Meta's latest LLM shows breakthrough reasoning and multilingual gains."),
        ("Vision Transformers Show Strong Zero-Shot Transfer", "2407.20299",
         "ViT models maintain performance across diverse visual domains."),
        ("Scaling Laws for Neural Language Models Revisited", "2407.20322",
         "Compute-optimal scaling reveals new LLM training-efficiency patterns."),
        ("RAG Improves Hallucination Control", "2407.18872",
         "Retrieval-augmented generation cuts factual errors across domains."),
        ("Self-Supervised Learning Sets New Benchmark Records", "2407.19234",
         "SSL rivals supervised approaches on ImageNet."),
        ("Graph Neural Networks for Molecular Property Prediction", "2407.18765",
         "GNNs beat traditional methods on drug-discovery benchmarks."),
        ("LoRA Fine-Tuning Nears Full-Model Performance", "2407.19223",
         "Low-rank adaptation cuts memory while keeping quality."),
        ("Multimodal Transformers Excel at Vision-Language Tasks", "2407.18934",
         "End-to-end models lead on VQA and captioning."),
        ("Continuous Learning Enables Lifelong Adaptation", "2407.19012",
         "New methods prevent catastrophic forgetting."),
        ("Sparse Attention Kernels Set Efficiency Records", "2407.18823",
         "Sparse patterns cut compute without hurting performance."),
    ]
    return [NewsItem("arxiv", t, f"https://arxiv.org/abs/{i}", s) for t, i, s in raw]


def discover_news() -> List[NewsItem]:
    """Fetch fresh AI/ML papers from arXiv RSS, with an offline fallback."""
    sources = [
        ("arxiv-cs-ai", "https://arxiv.org/rss/cs.AI"),
        ("arxiv-cs-lg", "https://arxiv.org/rss/cs.LG"),
    ]
    items: List[NewsItem] = []
    for sid, url in sources:
        try:
            print(f"   {sid}...", end="", flush=True)
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=10) as resp:
                items.extend(parse_rss_bytes(resp.read(), sid, limit=20))
            print(" ✓")
        except Exception:
            print(" ✗")
    if not items:
        print("   Using offline fallback data")
        items = _fallback_items()
    return items


print("\n🔍 Discovering AI/ML news...\n")
discovered_items = discover_news()
print(f"\n✅ Discovered {len(discovered_items)} items")


## Tools 2–4 — Rank, Assemble, Fallback (Gemini via `google.genai`)

These are the reasoning primitives the agent builds on:

- `call_gemini_json` — one modern `google.genai` call returning structured JSON.
- `keyword_rank` / `keyword_brief` — the deterministic offline **fallback**.
- `assemble_brief` — turns the model's chosen cards into a schema-valid
  `DailyBrief`, deduping, repairing URLs, and padding to exactly 10.

Nothing here is hard-wired into a fixed order — the *agent* (next section)
decides when to call the discovery tool and how to rank.


In [ ]:
# ── Reasoning primitives + deterministic fallback ────────────────────────────
_KEYWORDS = {
    3: ("model", "llm", "agent", "reasoning", "benchmark", "eval"),
    2: ("ai", "ml", "learning", "transformer", "multimodal"),
}


def _score(item: NewsItem) -> int:
    text = f"{item.title} {item.summary}".lower()
    return sum(pts for pts, kws in _KEYWORDS.items() if any(k in text for k in kws))


def keyword_rank(items: List[NewsItem], k: int = 10) -> List[NewsItem]:
    """Deterministic keyword ranking (offline fallback)."""
    return sorted(items, key=lambda x: (-_score(x), x.title.lower()))[:k]


def call_gemini_json(prompt: str) -> dict:
    """Single structured-JSON call through the modern google.genai SDK."""
    from google.genai import types
    resp = genai_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2,
            response_mime_type="application/json",
        ),
    )
    text = (resp.text or "").strip().replace("```json", "").replace("```", "").strip()
    return json.loads(text)


def _https(url: str) -> str:
    return url.replace("http://", "https://", 1) if url.startswith("http://") else url


def assemble_brief(cards_data: list, theme: str = "AI signal over noise") -> DailyBrief:
    """Turn model-chosen cards into a schema-valid 10-card DailyBrief."""
    by_url = {i.url: i for i in discovered_items}
    seen, cards = set(), []
    for c in cards_data:
        title = (c.get("title") or "").strip()
        url = _https((c.get("url") or "").strip())
        if not title or not url.startswith("https://") or title.lower() in seen:
            continue
        seen.add(title.lower())
        why = (c.get("why_it_matters") or "").strip()
        if not why:
            src = by_url.get(url)
            why = (src.summary[:200] if src and src.summary else "Key AI/ML development.")
        cards.append((title, url, why))
    # Pad from keyword ranking if the model returned fewer than 10 usable cards.
    for item in keyword_rank(discovered_items, k=len(discovered_items)):
        if len(cards) >= 10:
            break
        if item.title.lower() not in seen and item.url.startswith("https://"):
            seen.add(item.title.lower())
            cards.append((item.title, item.url,
                          item.summary[:200] or "Key AI/ML development."))
    cards = cards[:10]
    return DailyBrief(
        date=str(date.today()),
        theme=theme,
        cards=[BriefCard(i + 1, t, u, w) for i, (t, u, w) in enumerate(cards)],
    )


def keyword_brief(items: List[NewsItem]) -> DailyBrief:
    """Full deterministic brief — used when the LLM path is unavailable."""
    ranked = keyword_rank(items, k=10)
    return assemble_brief(
        [{"title": it.title, "url": it.url, "why_it_matters": it.summary[:200]}
         for it in ranked]
    )


print("✅ Reasoning primitives + fallback ready")


## The agent — real Google ADK (`Agent` + `Runner` + `FunctionTool`)

This is the core of the submission. We register the discovery function as an
ADK **`FunctionTool`** and hand it to a `google.adk.Agent`. The agent runs on
the official **`Runner`** with an **`InMemorySessionService`**, so it maintains
session state across turns — exactly the runtime the ADK codelabs use.

- **`curator_agent`** — decides to call `fetch_ai_news`, then reasons over the
  results to rank and explain the top 10 stories.
- **`critic_agent`** — reviews a finished brief and returns actionable fixes,
  powering the self-critique loop in the next section.

If ADK or the API key is unavailable, `ADK_AVAILABLE` stays `False` and the
run cell uses the deterministic fallback instead.


In [ ]:
# ── Real Google ADK: Agent + Runner + InMemorySessionService + FunctionTool ──
import uuid
import asyncio

ADK_AVAILABLE = False
curator_agent = critic_agent = None
try:
    from google.adk import Agent
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.adk.tools import FunctionTool
    from google.genai.types import Content, Part

    _session_service = InMemorySessionService()
    _APP = "ai_digest"
    _USER = "curator"

    def fetch_ai_news() -> str:
        """Fetch today's candidate AI/ML stories to rank.

        Returns:
            A JSON array of objects with `title`, `url`, and `summary`
            for every discovered story. Call this before ranking.
        """
        return json.dumps(
            [{"title": i.title, "url": i.url, "summary": i.summary}
             for i in discovered_items]
        )

    curator_agent = Agent(
        name="news_curator",
        model=GEMINI_MODEL,
        description="Curates the top 10 AI/ML stories into a daily brief.",
        instruction=(
            "You are an AI news curator. First call the fetch_ai_news tool to "
            "get today's candidate stories. Then choose the 10 most important "
            "and rank them (1 = most important). Prefer frontier models, agents, "
            "benchmarks, and results with real impact. Return ONLY JSON:\n"
            '{"theme": "<short theme>", "cards": [{"rank": 1, "title": "...", '
            '"url": "...", "why_it_matters": "<= 30 words, specific"}]}\n'
            "Use each story's URL exactly as provided. Return exactly 10 cards."
        ),
        tools=[FunctionTool(fetch_ai_news)],
    )

    critic_agent = Agent(
        name="brief_critic",
        model=GEMINI_MODEL,
        description="Reviews a daily brief and lists concrete improvements.",
        instruction=(
            "You are a strict editor reviewing a 10-card AI news brief. Check "
            "for: duplicate or near-duplicate stories, vague or generic "
            "why_it_matters text, and poor importance ordering. Return ONLY "
            'JSON: {"approved": true|false, "issues": ["..."], '
            '"suggested_order": ["<title>", ...]}. Approve only if the brief '
            "is genuinely high quality."
        ),
    )
    ADK_AVAILABLE = LLM_ENABLED
except Exception as e:
    print(f"⚠️  ADK unavailable ({str(e)[:70]}) — will use fallback")


async def _run_agent_async(agent, prompt: str) -> str:
    sid = str(uuid.uuid4())
    await _session_service.create_session(
        app_name=_APP, user_id=_USER, session_id=sid, state={}
    )
    runner = Runner(agent=agent, app_name=_APP, session_service=_session_service)
    message = Content(role="user", parts=[Part(text=prompt)])
    final = ""
    async for event in runner.run_async(
        user_id=_USER, session_id=sid, new_message=message
    ):
        if event.is_final_response() and event.content and event.content.parts:
            final = event.content.parts[0].text or ""
    return final


def run_adk(agent, prompt: str) -> str:
    """Synchronous wrapper so ADK runs inside the Kaggle notebook loop."""
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass
    return asyncio.get_event_loop().run_until_complete(
        _run_agent_async(agent, prompt)
    )


print(f"✅ ADK ready: {ADK_AVAILABLE}"
      + (f" — agents: {curator_agent.name}, {critic_agent.name}" if ADK_AVAILABLE else ""))


## Run — curate, then self-critique and revise

This is what makes it an *agent* rather than a pipeline. The curator produces a
brief; the critic reviews it and, if it isn't good enough, the curator revises
it using the critic's feedback. The loop runs up to `MAX_REVISIONS` times or
until the critic approves — real reason → critique → revise behaviour.

Every path ends in a schema-valid 10-card `DailyBrief`; if anything in the LLM
path fails, we fall back to deterministic keyword ranking.


In [ ]:
# ── Orchestrate: curate → self-critique → revise ─────────────────────────────
MAX_REVISIONS = 2


def _parse_json(text: str) -> dict:
    text = (text or "").strip().replace("```json", "").replace("```", "").strip()
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1:
        text = text[start:end + 1]
    return json.loads(text)


def brief_to_dict(b: DailyBrief) -> dict:
    return {
        "date": b.date, "theme": b.theme, "schema_version": b.schema_version,
        "cards": [{"rank": c.rank, "title": c.title, "url": c.url,
                   "why_it_matters": c.why_it_matters} for c in b.cards],
    }


def generate_brief():
    """Run the ADK agent with a self-critique loop; fall back if needed."""
    if not ADK_AVAILABLE:
        print("→ LLM path unavailable, using keyword fallback")
        return keyword_brief(discovered_items), "keyword-fallback"

    try:
        print("🤖 curator: fetching + ranking via ADK...")
        data = _parse_json(run_adk(curator_agent, "Produce today's AI Digest brief."))
        brief = assemble_brief(data.get("cards", []),
                               data.get("theme", "AI signal over noise"))

        for i in range(MAX_REVISIONS):
            print(f"🔎 critic: reviewing brief (round {i + 1})...")
            review = _parse_json(
                run_adk(critic_agent,
                        "Review this brief JSON:\n" + json.dumps(brief_to_dict(brief)))
            )
            if review.get("approved"):
                print("   ✅ critic approved")
                break
            issues = review.get("issues", [])
            print(f"   ✍️  revising: {len(issues)} issue(s)")
            fix = (
                "Revise the AI Digest brief to fix these issues, keeping the "
                "same JSON schema and exactly 10 cards:\n"
                + json.dumps(issues) + "\nCurrent brief:\n"
                + json.dumps(brief_to_dict(brief))
            )
            data = _parse_json(run_adk(curator_agent, fix))
            brief = assemble_brief(data.get("cards", []),
                                   data.get("theme", brief.theme))
        return brief, "adk-gemini"
    except Exception as e:
        print(f"⚠️  ADK path failed ({str(e)[:80]}), using keyword fallback")
        return keyword_brief(discovered_items), "keyword-fallback"


brief, brief_path = generate_brief()
print(f"\n✅ Generated DailyBrief via '{brief_path}' ({len(brief.cards)} cards)")


## Output & validation

Display the brief, export it as JSON, and assert the schema contract holds —
exactly 10 cards, ranks 1–10, HTTPS URLs, and non-empty explanations.


In [ ]:
# ── Display, export, validate ────────────────────────────────────────────────
print("=" * 70)
print(f"AI DIGEST — {brief.date}  |  theme: {brief.theme}  |  via: {brief_path}")
print("=" * 70)
for c in brief.cards:
    print(f"[{c.rank:2}] {c.title}")
    print(f"     {c.url}")
    print(f"     → {c.why_it_matters}")

# Export for downstream use / Kaggle output.
out = brief_to_dict(brief)
with open("brief_output.json", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print("\n💾 Wrote brief_output.json")

# Schema contract — fails loudly if the agent ever breaks the format.
assert len(brief.cards) == 10, "must have exactly 10 cards"
assert [c.rank for c in brief.cards] == list(range(1, 11)), "ranks must be 1..10"
assert all(c.url.startswith("https://") for c in brief.cards), "URLs must be HTTPS"
assert all(c.why_it_matters.strip() for c in brief.cards), "explanations required"
assert len({c.title.lower() for c in brief.cards}) == 10, "no duplicate titles"
print("✅ Schema validation passed (10 cards, ranks 1-10, HTTPS, no dupes)")


## How this maps to Google ADK

| Requirement | Where it lives |
|---|---|
| Official ADK runtime | `google.adk.Agent` + `Runner` + `InMemorySessionService` |
| Model-chosen tool call | `FunctionTool(fetch_ai_news)` on `curator_agent` |
| Modern LLM path | `google.genai` client, model `gemini-flash-latest` |
| Genuine agentic behaviour | curator → critic → revise self-critique loop |
| Safety net | deterministic keyword fallback, schema asserts |

**What changed from the earlier submission**

- Replaced the custom `ADKAgent` pipeline with the real ADK agent runtime.
- Dropped the deprecated `google.generativeai` / `gemini-pro` calls for the
  current `google.genai` SDK.
- Collapsed the triplicated Part A / B / C notebook into one clean build.
- Added a real reasoning loop instead of a fixed `discover → rank → validate`.

### Try it
- Swap in your own sources inside `discover_news()`.
- Raise `MAX_REVISIONS` to let the critic push the curator harder.
- Set `temperature` in `call_gemini_json` / agents to trade focus for variety.
